# 03 — Transformation
Spatially joins each listing to its Local Authority District (LAD) and MSOA, then joins all enrichment datasets. Writes enriched listing-level tables to the gold layer.

**Catalog:** `airbnb_app`  
**Reads from:** `airbnb_app.clean`  
**Writes to:** `airbnb_app.gold.airbnb_listings_{city}`  

| Step | What happens |
|------|-------------|
| A | Load LAD and MSOA boundary GeoJSON files |
| B | Load all enrichment tables from clean layer |
| C | Per city: spatial join LAD → spatial join MSOA → join enrichment → write to gold |

**Output schema per listing:** all clean listing columns + `lad_code`, `lad_name`, `msoa_code`, `msoa_name`, `median_house_price_2025`, `median_house_price_2015`, `price_growth_10yr`, `less_than_15_minute_walk`, `less_than_30_minute_walk`, `gp_surgery_count`, `total_parks_count`

> **Note:** Runs on the driver (single node) — GeoPandas cannot run on Spark workers.  
> **Coverage:** MSOA boundaries cover England and Wales only. Edinburgh gets `lad_code` but `msoa_code` will be null — Scotland uses Data Zones.  
> ⚠️ **Known Photon bug:** `UNION ALL` / `.show()` across all four city tables triggers a Photon compiler error when Edinburgh's all-null `msoa_code` is involved. Workaround: spot checks loop per-city. `.count()` and `.write` are unaffected.

## 0. Config

In [0]:
CLEAN_DB = "airbnb_app.clean"
GOLD_DB  = "airbnb_app.gold"

CITIES = ["london", "manchester", "edinburgh", "bristol"]

# LAD boundary file
LAD_BOUNDARY_PATH = "/Volumes/airbnb_app/raw/boundary_data/Local_Authority_Districts_December_2024_Boundaries_UK_BGC_-5461244619642504325.geojson"
LAD_CODE_COL = "LAD24CD"
LAD_NAME_COL = "LAD24NM"

# MSOA boundary file — England and Wales only
MSOA_BOUNDARY_PATH = "/Volumes/airbnb_app/raw/boundary_data/Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3_-4477917303172606123.geojson"
MSOA_CODE_COL = "MSOA21CD"
MSOA_NAME_COL = "MSOA21NM"

## 1. Setup

In [0]:
%pip install geopandas shapely --quiet

In [0]:
import os
import pandas as pd
import geopandas as gpd
from pyspark.sql import functions as F

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_DB}")
print(f"Schema ready: {GOLD_DB}")

gold_log = []

## 2. Helpers

In [0]:
def load_boundary(path: str, code_col: str, name_col: str, label: str) -> gpd.GeoDataFrame:
    """Load a GeoJSON boundary file, reproject to WGS84, and rename key columns."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"{label} boundary file not found at {path}")
    size_mb = os.path.getsize(path) / 1_048_576
    print(f"  ✓ {label}: {path} ({size_mb:.1f} MB)")

    gdf = gpd.read_file(path)

    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    elif gdf.crs.to_epsg() != 4326:
        print(f"    Reprojecting {label} from {gdf.crs} to EPSG:4326...")
        gdf = gdf.to_crs("EPSG:4326")

    if code_col not in gdf.columns or name_col not in gdf.columns:
        raise KeyError(
            f"Expected columns '{code_col}' / '{name_col}' not found in {label}.\n"
            f"Available: {list(gdf.columns)}"
        )

    prefix = label.lower()
    return gdf[[code_col, name_col, "geometry"]].rename(columns={
        code_col: f"{prefix}_code",
        name_col: f"{prefix}_name",
    })


def spatial_join(listings_pd: pd.DataFrame, boundary_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """Point-in-polygon join: assign boundary code/name to each listing."""
    listings_gdf = gpd.GeoDataFrame(
        listings_pd,
        geometry=gpd.points_from_xy(listings_pd["longitude"], listings_pd["latitude"]),
        crs="EPSG:4326",
    )
    joined = gpd.sjoin(listings_gdf, boundary_gdf, how="left", predicate="within")
    drop_cols = [c for c in ["geometry", "index_right"] if c in joined.columns]
    return joined.drop(columns=drop_cols)


def force_string_columns(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Force consistent nullable string typing on given columns.
    Edinburgh's msoa_code/msoa_name are entirely null after the spatial join.
    Without this, Spark may infer a different dtype for Edinburgh's table vs
    the other three cities, breaking UNION ALL queries downstream.
    """
    for col in columns:
        if col in df.columns:
            df[col] = df[col].astype("string").astype(object).where(df[col].notna(), None)
    return df


print("Helpers loaded.")

---
## Part A — Load Boundary Files

In [0]:
print("Loading boundary files...")
lad_gdf  = load_boundary(LAD_BOUNDARY_PATH,  LAD_CODE_COL,  LAD_NAME_COL,  "lad")
msoa_gdf = load_boundary(MSOA_BOUNDARY_PATH, MSOA_CODE_COL, MSOA_NAME_COL, "msoa")
print(f"  LAD polygons:  {len(lad_gdf):,}")
print(f"  MSOA polygons: {len(msoa_gdf):,} (England and Wales only)")

---
## Part B — Load Enrichment Tables

In [0]:
print("Loading enrichment tables...")

# MSOA-level enrichment — joined directly on msoa_code
house_prices_msoa_pd = spark.table(f"{CLEAN_DB}.house_prices_msoa").toPandas()
rail_msoa_pd         = spark.table(f"{CLEAN_DB}.amenities_rail_stations").toPandas()

# LAD-level enrichment — joined on lad_code
gp_pd    = spark.table(f"{CLEAN_DB}.amenities_gp_surgeries").toPandas()
parks_pd = spark.table(f"{CLEAN_DB}.amenities_parks").toPandas()

print(f"  House prices (MSOA):  {house_prices_msoa_pd.shape}")
print(f"  Rail stations (MSOA): {rail_msoa_pd.shape}")
print(f"  GP surgeries (LAD):   {gp_pd.shape}")
print(f"  Parks (LAD):          {parks_pd.shape}")

---
## Part C — Per-City Transformation Loop

In [0]:
for city in CITIES:
    print(f"\n{'='*50}")
    print(f"{city.upper()}")
    print(f"{'='*50}")

    src = f"{CLEAN_DB}.airbnb_listings_{city}"
    tgt = f"{GOLD_DB}.airbnb_listings_{city}"

    try:
        # Load clean listings
        listings_pd = spark.table(src).toPandas()
        print(f"  Loaded {len(listings_pd):,} listings")

        # Replace N/A strings with null before any operations
        listings_pd = listings_pd.replace("N/A", None)

        # Spatial join to LAD (UK-wide)
        listings_pd = spatial_join(listings_pd, lad_gdf)
        lad_matched = listings_pd["lad_code"].notna().sum()
        print(f"  LAD matched:  {lad_matched:,} / {len(listings_pd):,}")

        # Spatial join to MSOA (England/Wales only — Edinburgh will be null)
        listings_pd = spatial_join(listings_pd, msoa_gdf)
        msoa_matched = listings_pd["msoa_code"].notna().sum()
        print(f"  MSOA matched: {msoa_matched:,} / {len(listings_pd):,}")

        # Join house prices on msoa_code — native MSOA granularity
        listings_pd = listings_pd.merge(
            house_prices_msoa_pd[["msoa_code", "median_house_price_2025",
                                   "median_house_price_2015", "price_growth_10yr"]],
            on="msoa_code", how="left"
        )

        # Join rail walk-time on msoa_code — native MSOA granularity
        listings_pd = listings_pd.merge(
            rail_msoa_pd[["msoa_code", "less_than_15_minute_walk",
                          "less_than_30_minute_walk", "less_than_60_minute_walk"]],
            on="msoa_code", how="left"
        )

        # Join GP surgeries on lad_code — LAD-level only
        listings_pd = listings_pd.merge(
            gp_pd[["lad_code", "gp_surgery_count", "gps_per_100000_people"]],
            on="lad_code", how="left"
        )

        # Join parks on lad_code — LAD-level only
        listings_pd = listings_pd.merge(
            parks_pd[["lad_code", "total_parks_count",
                      "parks_and_play_areas_per_100000_people"]],
            on="lad_code", how="left"
        )

        # Cast float32 → float64 — known Arrow serialization issue
        for c in listings_pd.select_dtypes(include="float32").columns:
            listings_pd[c] = listings_pd[c].astype("float64")

        # Force consistent string typing on join key columns
        # Edinburgh's msoa_code/msoa_name are entirely null — without this
        # Spark may infer a different dtype, breaking downstream UNION ALL
        listings_pd = force_string_columns(
            listings_pd, ["lad_code", "lad_name", "msoa_code", "msoa_name"]
        )

        # Write to gold
        listings_spark = spark.createDataFrame(listings_pd)
        listings_spark = listings_spark.withColumn("_gold_created_at", F.current_timestamp())

        (
            listings_spark.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(tgt)
        )

        row_count = listings_spark.count()
        print(f"  ✓ {tgt} ({row_count:,} rows)")
        gold_log.append({
            "city": city, "status": "ok", "rows": row_count,
            "lad_matched": int(lad_matched), "msoa_matched": int(msoa_matched)
        })

    except Exception as e:
        print(f"  ✗ {tgt} — {e}")
        gold_log.append({"city": city, "status": "error", "error": str(e)})

## D. Transformation summary

In [0]:
summary = pd.DataFrame(gold_log)
display(summary)

failures = summary[summary["status"] == "error"]
if not failures.empty:
    raise RuntimeError(f"Transformation failed for:\n{failures[['city', 'error']].to_string()}")

print("\nTransformation complete.")

## E. Spot checks

> Spot checks run per-city to avoid the Photon UNION ALL bug — see note at top of notebook.

In [0]:
# Bristol: confirm multiple MSOAs (not just one LAD row)
spark.sql("""
    SELECT msoa_name, COUNT(*) AS listing_count,
           ROUND(AVG(median_house_price_2025), 0) AS avg_house_price
    FROM airbnb_app.gold.airbnb_listings_bristol
    WHERE msoa_code IS NOT NULL
    GROUP BY msoa_name
    ORDER BY listing_count DESC
    LIMIT 10
""").display()

In [0]:
# Manchester: confirm LAD, MSOA, house price, and rail walk-time all populated
spark.sql("""
    SELECT id, lad_name, msoa_name, median_house_price_2025,
           less_than_15_minute_walk, gp_surgery_count, total_parks_count
    FROM airbnb_app.gold.airbnb_listings_manchester
    LIMIT 5
""").display()

In [0]:
# Match rate per city
# Edinburgh: lad_code populated, msoa_code and house price null (expected)
for city in CITIES:
    print(f"\n{city.upper()}")
    spark.sql(f"""
        SELECT
            COUNT(*)                                                  AS total_listings,
            SUM(CASE WHEN lad_code IS NULL THEN 1 END)                AS missing_lad,
            SUM(CASE WHEN msoa_code IS NULL THEN 1 END)               AS missing_msoa,
            SUM(CASE WHEN median_house_price_2025 IS NULL THEN 1 END) AS missing_house_price
        FROM airbnb_app.gold.airbnb_listings_{city}
    """).display()

## Notes

- **Two spatial joins per listing:** LAD (UK-wide) and MSOA (England and Wales only). Both run on the driver via GeoPandas.
- **House prices and rail stations join at MSOA level** — no aggregation, giving genuine neighbourhood-level differentiation. This fixes Bristol showing only one row at LAD level.
- **GP surgeries and parks join at LAD level** — these source datasets do not go finer than LAD, so the same value repeats across all MSOAs within a LAD.
- **Edinburgh:** `lad_code` populates correctly (LAD boundaries are UK-wide). `msoa_code`, house prices, and rail data are all null since those sources cover England and Wales only. Edinburgh house price and rent data is handled in the next notebook via a separate manual file.
- **Next step:** Run `04_aggregation_export.ipynb`.